# Data Augmentation Pipeline — `chord_data_1217`

This notebook generates augmented versions of all audio files in `chord_data_1217` and produces the corresponding `.jams` annotation files for each augmentation.

**Augmentation types applied:**
- Convolutional reverb (IR-based)
- Real-world noise from FreeSound (café, street, wind)
- Random EQ
- Dynamic compression

**Output structure:**
```
final_augmentation/
├── data_augmentation_files/   ← augmented .mp3 audio files
└── data_augmentation_jams/    ← copied .jams (one per augmented audio)
```

---
## ⚙️ Step 0 — Install dependencies

In [1]:
!pip install git+https://github.com/MTG/freesound-python.git librosa soundfile scipy numpy jams

  Cloning https://github.com/MTG/freesound-python.git to C:\Users\User\AppData\Local\Temp\pip-req-build-e_f1c0ab
  Resolved https://github.com/MTG/freesound-python.git to commit 73cf6d14f7ce8174d943bdc78ff30f99878a5db8
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'


  Running command git clone --filter=blob:none --quiet https://github.com/MTG/freesound-python.git 'C:\Users\User\AppData\Local\Temp\pip-req-build-e_f1c0ab'


---
## 🔑 Step 1 — FreeSound API authentication

### How to get your API credentials:
1. Log in at [https://freesound.org](https://freesound.org) with your UPF account.
2. Go to **your username → Settings → API credentials** (or directly to [https://freesound.org/apiv2/apply/](https://freesound.org/apiv2/apply/)).
3. Create a new application — name it anything (e.g. `ACE_augmentation`).
4. Copy your **Client ID** and **Client Secret**.
5. For this notebook we use the **Client Credentials flow** (no user login needed), which gives you a token to download sounds with a Creative Commons license.

> ⚠️ **Never commit your credentials to git.** Use environment variables or a local `.env` file.

In [2]:
import os

# ── Paste your credentials here (or load from environment) ──────────────────
FREESOUND_API_KEY = os.environ.get("FREESOUND_API_KEY", "H7F3clGsFSWXhlEUKfiIJtkxCy9nYXfkdUfmIqEQ")
# ─────────────────────────────────────────────────────────────────────────────

import freesound
fs_client = freesound.FreesoundClient()
fs_client.set_token(FREESOUND_API_KEY, "token")
print("FreeSound client ready.")

FreeSound client ready.


c:\UPF\MIR\MirInharmonicAugmentation\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(


---
## 📁 Step 2 — Project paths

In [3]:
from pathlib import Path

# ── Adjust these paths to your local setup ───────────────────────────────────
CHORD_DATA_DIR   = Path("chord_data_1217")
AUDIO_DIR        = CHORD_DATA_DIR / "audio"
JAMS_DIR         = CHORD_DATA_DIR / "references_v2"
IR_DIR           = Path("irs")           # folder with your impulse responses
VOCAB_PATH       = Path("final_augmentation\chords_vocab.joblib") # path to chord vocabulary

OUTPUT_ROOT      = Path("final_augmentation")
OUTPUT_AUDIO     = OUTPUT_ROOT / "data_augmentation_files"
OUTPUT_JAMS      = OUTPUT_ROOT / "data_augmentation_jams"
NOISE_CACHE_DIR  = OUTPUT_ROOT / "noise_cache"  # downloaded FreeSound clips

for d in [OUTPUT_AUDIO, OUTPUT_JAMS, NOISE_CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Paths OK.")
print(f"  Audio dir   : {AUDIO_DIR}")
print(f"  JAMS dir    : {JAMS_DIR}")
print(f"  Output audio: {OUTPUT_AUDIO}")
print(f"  Output JAMS : {OUTPUT_JAMS}")

Paths OK.
  Audio dir   : chord_data_1217\audio
  JAMS dir    : chord_data_1217\references_v2
  Output audio: final_augmentation\data_augmentation_files
  Output JAMS : final_augmentation\data_augmentation_jams


---
## 🔍 Step 3 — Discover audio files

In [4]:
audio_files = sorted(AUDIO_DIR.rglob("*.mp3"))
print(f"Found {len(audio_files)} audio files.")
for f in audio_files[:5]:
    print(" ", f.name)

Found 1217 audio files.
  TR6R91L11C8A40D710.mp3
  TRACGVT149E3B9BE3F.mp3
  TRACPPB149E33C10B9.mp3
  TRADINA127F847B84E.mp3
  TRAEQJQ149E3BA694B.mp3


---
## 🌐 Step 4 — Download background noises from FreeSound

We download a small pool of clips per category. Each clip is saved to `noise_cache/`.
Clips are concatenated/trimmed at augmentation time to match the target audio length.

### FreeSound search tags used:
| Category | Tags |
|---|---|
| Café | `café ambience`, `coffee shop noise` |
| Street / traffic | `city street`, `traffic noise` |
| Wind | `wind outdoor`, `wind noise` |

In [9]:
import requests
import time

NOISE_QUERIES = {
    "cafe":   ["cafe ambience", "coffee shop background noise"],
    "street": ["city street ambience", "traffic noise urban"],
    "wind":   ["wind outdoor ambience", "wind noise exterior"],
}

# Number of clips to download per category (adjust to taste)
CLIPS_PER_CATEGORY = 3


def download_freesound_clips(
    client: freesound.FreesoundClient,
    queries: dict[str, list[str]],
    output_dir: Path,
    clips_per_category: int = 3,
    min_duration: float = 10.0,   # seconds — skip very short clips
    max_duration: float = 300.0,  # seconds
) -> dict[str, list[Path]]:
    """Search FreeSound and download HQ-preview MP3s for each noise category.
    Uses client.search() which is the correct method in the freesound PyPI package.
    Returns a dict mapping category -> list of local paths.
    """
    category_files: dict[str, list[Path]] = {cat: [] for cat in queries}

    for category, query_list in queries.items():
        cat_dir = output_dir / category
        cat_dir.mkdir(parents=True, exist_ok=True)
        collected = 0

        for query in query_list:
            if collected >= clips_per_category:
                break
            print(f"[{category}] Searching: '{query}'...")
            try:
                # freesound PyPI package exposes client.search()
                results = client.search(
                    query=query,
                    filter=f"duration:[{min_duration} TO {max_duration}]",
                    fields="id,name,duration,previews,license",
                    page_size=10,
                )
            except Exception as e:
                print(f"  Search error: {e}")
                continue

            for sound in results:
                if collected >= clips_per_category:
                    break

                dest_mp3 = cat_dir / f"{category}_{sound.id}.mp3"
                if dest_mp3.exists():
                    print(f"  [cached] {dest_mp3.name}")
                    category_files[category].append(dest_mp3)
                    collected += 1
                    continue

                # HQ preview does not require OAuth — just the API key token
                try:
                    preview_url = sound.previews.preview_hq_mp3
                    r = requests.get(
                        preview_url,
                        headers={"Authorization": f"Token {FREESOUND_API_KEY}"},
                        timeout=30,
                    )
                    r.raise_for_status()
                    dest_mp3.write_bytes(r.content)
                    print(f"  [downloaded] {dest_mp3.name}  ({sound.duration:.1f}s)")
                    category_files[category].append(dest_mp3)
                    collected += 1
                    time.sleep(0.3)  # be polite to the API
                except Exception as e:
                    print(f"  Download error for sound {sound.id}: {e}")

    return category_files


noise_files = download_freesound_clips(
    fs_client, NOISE_QUERIES, NOISE_CACHE_DIR, clips_per_category=CLIPS_PER_CATEGORY
)

for cat, files in noise_files.items():
    print(f"{cat}: {len(files)} clip(s) available")

[cafe] Searching: 'cafe ambience'...
  [downloaded] cafe_366483.mp3  (241.0s)
  [downloaded] cafe_363713.mp3  (43.0s)
  [downloaded] cafe_516435.mp3  (13.6s)
[street] Searching: 'city street ambience'...
  [downloaded] street_622736.mp3  (109.3s)
  [downloaded] street_608157.mp3  (123.9s)
  [downloaded] street_723607.mp3  (87.8s)
[wind] Searching: 'wind outdoor ambience'...
  [downloaded] wind_138294.mp3  (15.8s)
  [downloaded] wind_326908.mp3  (131.0s)
  [downloaded] wind_617747.mp3  (130.1s)
cafe: 3 clip(s) available
street: 3 clip(s) available
wind: 3 clip(s) available


---
## 🎛️ Step 5 — Augmentation configuration

All wet/dry ratios default to **0.5 (50 % wet / 50 % dry)**.

In [10]:
import random
from pathlib import Path

SR = 22050  # sample rate used throughout

# Impulse responses available locally
ir_files = sorted(IR_DIR.rglob("*.wav")) + sorted(IR_DIR.rglob("*.flac"))
print(f"Found {len(ir_files)} impulse response file(s).")

# Build augmentation plan: list of (suffix, function_name, args_dict)
# Each entry produces one output file per source audio.

AUGMENTATION_PLAN = []

# -- Reverb: one augmentation per IR file
for ir_path in ir_files:
    import re
    slug = re.sub(r"[^A-Za-z0-9]+", "-", ir_path.stem).strip("-").lower()
    AUGMENTATION_PLAN.append((
        f"reverb_{slug}",
        "apply_reverb",
        {"ir_path": str(ir_path), "sr": SR, "wet_dry_mix": 0.5},
    ))

# -- Real noise: one augmentation per category (random clip chosen at runtime)
for category, files in noise_files.items():
    if files:
        AUGMENTATION_PLAN.append((
            f"noise_{category}",
            "add_real_noise",
            {"noise_files": files, "sr": SR, "wet_dry_mix": 0.5},
        ))

# -- EQ (single pass, random gains)
AUGMENTATION_PLAN.append((
    "eq",
    "randomly_eq",
    {"sr": SR, "gain_range": (-6, 6)},
))

# -- Compression
AUGMENTATION_PLAN.append((
    "compression",
    "apply_compression",
    {"sr": SR, "threshold_db": -20, "ratio": 4.0, "attack_ms": 5, "release_ms": 100},
))

print(f"\nAugmentation plan: {len(AUGMENTATION_PLAN)} augmentation type(s)")
for suffix, fn, _ in AUGMENTATION_PLAN:
    print(f"  [{fn}]  suffix → _{suffix}")

Found 34 impulse response file(s).

Augmentation plan: 39 augmentation type(s)
  [apply_reverb]  suffix → _reverb_1st-baptist-nashville-balcony
  [apply_reverb]  suffix → _reverb_1st-baptist-nashville-balcony
  [apply_reverb]  suffix → _reverb_1st-baptist-nashville-far-close
  [apply_reverb]  suffix → _reverb_1st-baptist-nashville-far-wide
  [apply_reverb]  suffix → _reverb_crash-ir
  [apply_reverb]  suffix → _reverb_hh-ir
  [apply_reverb]  suffix → _reverb_kick-ir
  [apply_reverb]  suffix → _reverb_ride-ir
  [apply_reverb]  suffix → _reverb_snare-ir
  [apply_reverb]  suffix → _reverb_toma-ir
  [apply_reverb]  suffix → _reverb_tomb-ir
  [apply_reverb]  suffix → _reverb_tomc-ir
  [apply_reverb]  suffix → _reverb_average-space-ir-0
  [apply_reverb]  suffix → _reverb_impulseresponseheslingtonchurch-001
  [apply_reverb]  suffix → _reverb_impulseresponseheslingtonchurch-002
  [apply_reverb]  suffix → _reverb_impulseresponseheslingtonchurch-003
  [apply_reverb]  suffix → _reverb_impulserespo

---
## 🚀 Step 6 — Run augmentation pipeline

For each source audio file and each augmentation type:
1. Generate `<stem>_<suffix>.mp3` in `data_augmentation_files/`
2. Copy the corresponding `.jams` to `data_augmentation_jams/<stem>_<suffix>.jams`

In [11]:
import shutil
import random
import traceback
from data_augmentation_merged import (
    apply_reverb,
    add_real_noise,
    randomly_eq,
    apply_compression,
)

AUGMENTATION_FN_MAP = {
    "apply_reverb":      apply_reverb,
    "add_real_noise":    add_real_noise,
    "randomly_eq":       randomly_eq,
    "apply_compression": apply_compression,
}


def get_jams_path(audio_path: Path, jams_dir: Path) -> Path | None:
    """Return the .jams file that corresponds to an audio file, or None."""
    candidate = jams_dir / (audio_path.stem + ".jams")
    return candidate if candidate.exists() else None


def resolve_args(args: dict) -> dict:
    """For add_real_noise: pick one random clip from the pool at runtime."""
    resolved = dict(args)
    if "noise_files" in resolved:
        resolved["noise_path"] = str(random.choice(resolved.pop("noise_files")))
    return resolved


ok_count = 0
skip_count = 0
error_count = 0

for audio_path in audio_files:
    jams_src = get_jams_path(audio_path, JAMS_DIR)
    if jams_src is None:
        print(f"[WARN] No .jams found for {audio_path.name} — skipping.")
        skip_count += 1
        continue

    for suffix, fn_name, base_args in AUGMENTATION_PLAN:
        out_stem     = f"{audio_path.stem}_{suffix}"
        out_audio    = OUTPUT_AUDIO / f"{out_stem}.mp3"
        out_jams     = OUTPUT_JAMS  / f"{out_stem}.jams"

        # Skip if both outputs already exist (resume-friendly)
        if out_audio.exists() and out_jams.exists():
            skip_count += 1
            continue

        try:
            fn = AUGMENTATION_FN_MAP[fn_name]
            args = resolve_args(base_args)
            fn(str(audio_path), str(out_audio), args)

            # Copy .jams with new name
            shutil.copy2(jams_src, out_jams)

            print(f"[OK] {out_audio.name}")
            ok_count += 1

        except Exception as e:
            print(f"[ERROR] {out_stem}: {e}")
            traceback.print_exc()
            error_count += 1

print(f"\n=== Pipeline summary ===")
print(f"  Generated : {ok_count}")
print(f"  Skipped   : {skip_count}")
print(f"  Errors    : {error_count}")

[OK] TR6R91L11C8A40D710_reverb_1st-baptist-nashville-balcony.mp3
[OK] TR6R91L11C8A40D710_reverb_1st-baptist-nashville-far-close.mp3
[OK] TR6R91L11C8A40D710_reverb_1st-baptist-nashville-far-wide.mp3
[OK] TR6R91L11C8A40D710_reverb_crash-ir.mp3
[OK] TR6R91L11C8A40D710_reverb_hh-ir.mp3
[OK] TR6R91L11C8A40D710_reverb_kick-ir.mp3
[OK] TR6R91L11C8A40D710_reverb_ride-ir.mp3
[OK] TR6R91L11C8A40D710_reverb_snare-ir.mp3
[OK] TR6R91L11C8A40D710_reverb_toma-ir.mp3
[OK] TR6R91L11C8A40D710_reverb_tomb-ir.mp3
[OK] TR6R91L11C8A40D710_reverb_tomc-ir.mp3
[OK] TR6R91L11C8A40D710_reverb_average-space-ir-0.mp3
[OK] TR6R91L11C8A40D710_reverb_impulseresponseheslingtonchurch-001.mp3
[OK] TR6R91L11C8A40D710_reverb_impulseresponseheslingtonchurch-002.mp3
[OK] TR6R91L11C8A40D710_reverb_impulseresponseheslingtonchurch-003.mp3
[OK] TR6R91L11C8A40D710_reverb_impulseresponseheslingtonchurch-004.mp3
[OK] TR6R91L11C8A40D710_reverb_impulseresponseheslingtonchurch-005.mp3
[OK] TR6R91L11C8A40D710_reverb_impulseresponsehes

KeyboardInterrupt: 

---
## ✅ Step 7 — Verification

Check that every audio file in the output has a matching `.jams`.

In [ ]:
aug_audio_files = sorted(OUTPUT_AUDIO.glob("*.mp3"))
aug_jams_files  = sorted(OUTPUT_JAMS.glob("*.jams"))

audio_stems = {f.stem for f in aug_audio_files}
jams_stems  = {f.stem for f in aug_jams_files}

missing_jams  = audio_stems - jams_stems
orphan_jams   = jams_stems  - audio_stems

print(f"Augmented audio files : {len(aug_audio_files)}")
print(f"Augmented JAMS files  : {len(aug_jams_files)}")

if missing_jams:
    print(f"\n⚠️  Audio files WITHOUT matching .jams ({len(missing_jams)}):")
    for s in sorted(missing_jams): print(f"  {s}")
else:
    print("\n✅ Every audio file has a matching .jams.")

if orphan_jams:
    print(f"\n⚠️  Orphan .jams without audio ({len(orphan_jams)}):")
    for s in sorted(orphan_jams): print(f"  {s}")

---
## 📋 Step 8 — Generate filelist.txt for augmented audio

Same format as the original `audio/filelist.txt`.

In [ ]:
filelist_path = OUTPUT_AUDIO / "filelist.txt"
with open(filelist_path, "w") as f:
    for audio_file in sorted(OUTPUT_AUDIO.glob("*.mp3")):
        f.write(audio_file.name + "\n")

print(f"filelist.txt written → {filelist_path}")
print(f"  {len(sorted(OUTPUT_AUDIO.glob('*.mp3')))} entries")